In [15]:
import torch
import os
from PIL import Image
from torchvision import transforms
from transformers import SegformerForSemanticSegmentation
import numpy as np

# 设置路径
image_dir = "D:/2024/paper2/model/input_images2/zuoshu"  # 验证图像文件夹
output_mask_dir = "D:/2024/paper2/model/input_masks/zuoshu"  # 输出掩码文件夹
model_dir = "D:/2024/paper2/model/Model/zuoshu"  # 模型文件夹

# 加载完整模型
model_path = os.path.join(model_dir, "segformer_full_model.pth")
model = torch.load(model_path)  # 直接加载完整模型
model.eval()  # 设置为评估模式

# 数据预处理（与训练时相同）
transform = transforms.Compose([
    transforms.ToTensor(),  # 转换为 Tensor
])

# 创建输出文件夹（如果不存在）
os.makedirs(output_mask_dir, exist_ok=True)

# 验证集图像文件名
image_filenames = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# 推理过程
with torch.no_grad():  # 在推理时不需要计算梯度
    for filename in image_filenames:
        img_path = os.path.join(image_dir, filename)
        
        # 读取图像
        image = Image.open(img_path).convert("RGB")
        
        # 保存原始尺寸
        original_size = image.size
        
        # 统一尺寸（假设训练时输入尺寸是1024x1024）
        target_size = (1024, 1024)  # 如果你的验证图像不是1024x1024，可能需要调整此处
        image_resized = image.resize(target_size)

        # 进行预处理
        image_tensor = transform(image_resized).unsqueeze(0).to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

        # 模型推理
        outputs = model(image_tensor).logits

        # 获取预测的类别标签（背景=0, 树干=1）
        predicted_mask = torch.argmax(outputs, dim=1).squeeze(0).cpu().numpy().astype(np.uint8)

        # 将输出的掩码转换为 0 和 255（黑色背景，白色树干）
        predicted_mask = predicted_mask * 255  # 0 -> 0, 1 -> 255

        # 将预测掩码调整回原始输入图像的大小
        predicted_mask_resized = Image.fromarray(predicted_mask)
        predicted_mask_resized = predicted_mask_resized.resize(original_size, Image.NEAREST)

        # 保存掩码图像
        mask_filename = os.path.join(output_mask_dir, filename.replace(".jpg", ".png").replace(".jpeg", ".png").replace(".png", ".png"))
        predicted_mask_resized.save(mask_filename)

        print(f"Saved mask for {filename} to {mask_filename}")

print("推理完成，所有掩码已保存！")

C:\Users\admin\AppData\Local\Temp\ipykernel_18884\3795042778.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path)  # 直接加载完整模型


Saved mask for 09080040006000_04.jpg to D:/2024/paper2/model/input_masks/zuoshu\09080040006000_04.png
Saved mask for 09080040006000_05.jpg to D:/2024/paper2/model/input_masks/zuoshu\09080040006000_05.png
Saved mask for 09080080006000_01.jpg to D:/2024/paper2/model/input_masks/zuoshu\09080080006000_01.png
Saved mask for 09080080006000_02.jpg to D:/2024/paper2/model/input_masks/zuoshu\09080080006000_02.png
Saved mask for 09080080006000_03.jpg to D:/2024/paper2/model/input_masks/zuoshu\09080080006000_03.png
Saved mask for 09080080006000_05.jpg to D:/2024/paper2/model/input_masks/zuoshu\09080080006000_05.png
Saved mask for 09080080019000_03.jpg to D:/2024/paper2/model/input_masks/zuoshu\09080080019000_03.png
Saved mask for 09080080019000_04.jpg to D:/2024/paper2/model/input_masks/zuoshu\09080080019000_04.png
Saved mask for 09080080019000_05.jpg to D:/2024/paper2/model/input_masks/zuoshu\09080080019000_05.png
Saved mask for 09080080020000_03.jpg to D:/2024/paper2/model/input_masks/zuoshu\09

In [16]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
from PIL import Image

# 使用GPU或CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

###################################
# 定义生成器（U-Net风格）
###################################
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        def down(in_channels, out_channels, normalize=True):
            layers = [nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False)]
            if normalize:
                layers.append(nn.InstanceNorm2d(out_channels))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        def up(in_channels, out_channels, dropout=0.0):
            layers = [
                nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
                nn.InstanceNorm2d(out_channels),
                nn.ReLU(inplace=True)
            ]
            if dropout:
                layers.append(nn.Dropout(dropout))
            return layers

        self.down1 = nn.Sequential(*down(1, 64, normalize=False))
        self.down2 = nn.Sequential(*down(64, 128))
        self.down3 = nn.Sequential(*down(128, 256))
        self.down4 = nn.Sequential(*down(256, 512))
        self.down5 = nn.Sequential(*down(512, 512))
        self.down6 = nn.Sequential(*down(512, 512))
        self.down7 = nn.Sequential(*down(512, 512))
        self.down8 = nn.Sequential(*down(512, 512, normalize=False))

        self.up1 = nn.Sequential(*up(512, 512, dropout=0.5))
        self.up2 = nn.Sequential(*up(1024, 512, dropout=0.5))
        self.up3 = nn.Sequential(*up(1024, 512, dropout=0.5))
        self.up4 = nn.Sequential(*up(1024, 512))
        self.up5 = nn.Sequential(*up(1024, 256))
        self.up6 = nn.Sequential(*up(512, 128))
        self.up7 = nn.Sequential(*up(256, 64))
        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, 1, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        d5 = self.down5(d4)
        d6 = self.down6(d5)
        d7 = self.down7(d6)
        d8 = self.down8(d7)

        u1 = self.up1(d8)
        u1 = torch.cat([u1, d7], 1)
        u2 = self.up2(u1)
        u2 = torch.cat([u2, d6], 1)
        u3 = self.up3(u2)
        u3 = torch.cat([u3, d5], 1)
        u4 = self.up4(u3)
        u4 = torch.cat([u4, d4], 1)
        u5 = self.up5(u4)
        u5 = torch.cat([u5, d3], 1)
        u6 = self.up6(u5)
        u6 = torch.cat([u6, d2], 1)
        u7 = self.up7(u6)
        u7 = torch.cat([u7, d1], 1)
        return self.final(u7)

###################################
# 判别器（PatchGAN）
###################################
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()

        def disc_block(in_channels, out_channels, normalize=True):
            layers = [nn.Conv2d(in_channels, out_channels, 4, 2, 1, bias=False)]
            if normalize:
                layers.append(nn.InstanceNorm2d(out_channels))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *disc_block(2, 64, normalize=False),
            *disc_block(64, 128),
            *disc_block(128, 256),
            nn.Conv2d(256, 1, 4, 1, 1, bias=False)
        )

    def forward(self, x, y):
        inp = torch.cat([x, y], dim=1)
        return self.model(inp)

###################################
# 数据集定义
###################################
class MaskDataset(Dataset):
    def __init__(self, input_dir, target_dir, transform=None):
        self.input_dir = input_dir
        self.target_dir = target_dir
        self.input_images = sorted([f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        self.target_images = sorted([f for f in os.listdir(target_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        self.transform = transform

    def __len__(self):
        return len(self.input_images)

    def __getitem__(self, idx):
        inp_path = os.path.join(self.input_dir, self.input_images[idx])
        tgt_path = os.path.join(self.target_dir, self.target_images[idx])

        inp_img = Image.open(inp_path).convert('L')
        tgt_img = Image.open(tgt_path).convert('L')

        if self.transform:
            inp_img = self.transform(inp_img)
            tgt_img = self.transform(tgt_img)

        return inp_img, tgt_img

###################################
# 图像变换
###################################
transform = transforms.Compose([
    transforms.Resize((256, 256)), 
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

###################################
# 数据加载
###################################
input_dir = 'D:/2024/paper2/model/input_masks/zuoshu'
target_dir = 'D:/2024/paper2/model/output_masks/zuoshu'

dataset = MaskDataset(input_dir, target_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

###################################
# 初始化模型和优化器
###################################
generator = Generator().to(device)
discriminator = Discriminator().to(device)

optim_G = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optim_D = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

criterion_GAN = nn.BCEWithLogitsLoss()
criterion_L1 = nn.L1Loss()

###################################
# 图像保存函数
###################################
def save_image(tensor, path):
    tensor = tensor.cpu().clone()
    tensor = (tensor * 0.5) + 0.5  # [-1,1] -> [0,1]
    arr = tensor.squeeze().numpy() * 255  # 去掉所有长度为1的维度
    arr = arr.astype('uint8')
    img = Image.fromarray(arr, mode='L')
    img.save(path)


###################################
# 训练函数
###################################
def train(num_epochs=100, checkpoint_dir='D:/2024/paper2/model/GAN/zuoshu'):
    os.makedirs(checkpoint_dir, exist_ok=True)
    for epoch in range(num_epochs):
        for i, (inp, tgt) in enumerate(dataloader):
            inp, tgt = inp.to(device), tgt.to(device)

            # ---------------------
            # 训练判别器
            # ---------------------
            optim_D.zero_grad()
            fake_tgt = generator(inp)
            pred_real = discriminator(inp, tgt)
            pred_fake = discriminator(inp, fake_tgt.detach())

            valid = torch.ones_like(pred_real, device=device)
            fake = torch.zeros_like(pred_real, device=device)

            loss_D_real = criterion_GAN(pred_real, valid)
            loss_D_fake = criterion_GAN(pred_fake, fake)
            loss_D = (loss_D_real + loss_D_fake) * 0.5
            loss_D.backward()
            optim_D.step()

            # ---------------------
            # 训练生成器
            # ---------------------
            optim_G.zero_grad()
            pred_fake = discriminator(inp, fake_tgt)
            loss_G_GAN = criterion_GAN(pred_fake, valid)
            loss_G_L1 = criterion_L1(fake_tgt, tgt) * 100.0

            loss_G = loss_G_GAN + loss_G_L1
            loss_G.backward()
            optim_G.step()

            if i % 50 == 0:
                print(f"Epoch[{epoch+1}/{num_epochs}] Step[{i}/{len(dataloader)}], "
                      f"D Loss: {loss_D.item():.4f}, G Loss: {loss_G.item():.4f}")

        torch.save(generator.state_dict(), os.path.join(checkpoint_dir, f'generator_epoch_{epoch+1}.pth'))
        torch.save(discriminator.state_dict(), os.path.join(checkpoint_dir, f'discriminator_epoch_{epoch+1}.pth'))

    # 保存最终模型
    torch.save(generator.state_dict(), os.path.join(checkpoint_dir, 'generator_final.pth'))
    torch.save(discriminator.state_dict(), os.path.join(checkpoint_dir, 'discriminator_final.pth'))
    print("训练完成并已保存最终模型参数。")

###################################
# 测试函数：对单张图片进行补全
###################################
def generate_complete_mask(input_image_path, output_path, generator_path=None):
    if generator_path is not None and os.path.exists(generator_path):
        generator.load_state_dict(torch.load(generator_path, map_location=device))
        generator.eval()
        print(f"已加载生成器模型：{generator_path}")
    else:
        generator.eval()

    with torch.no_grad():
        img = Image.open(input_image_path).convert('L')
        img = transform(img).unsqueeze(0).to(device)
        fake_tgt = generator(img)
        save_image(fake_tgt, output_path)
    print(f"生成结果已保存至：{output_path}")

###################################
# 批量处理测试文件夹中的所有图像
###################################
def generate_complete_masks(input_dir, output_dir, generator_path=None):
    if generator_path is not None and os.path.exists(generator_path):
        generator.load_state_dict(torch.load(generator_path, map_location=device))
        generator.eval()
        print(f"已加载生成器模型：{generator_path}")
    else:
        generator.eval()

    os.makedirs(output_dir, exist_ok=True)
    image_files = sorted([f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))])

    for filename in image_files:
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        with torch.no_grad():
            img = Image.open(input_path).convert('L')
            img = transform(img).unsqueeze(0).to(device)
            fake_tgt = generator(img)
            save_image(fake_tgt, output_path)

        print(f"已处理：{input_path} -> {output_path}")

###################################
# 主程序执行训练和批量生成
###################################
if __name__ == "__main__":
    # 训练模型
    train(num_epochs=100, checkpoint_dir='D:/2024/paper2/model/GAN/zuoshu')

    # 加载最终生成器模型，批量处理所有测试图像
    final_generator_path = 'D:/2024/paper2/model/GAN/zuoshu/generator_final.pth'
    test_input_dir = 'D:/2024/paper2/model/input_masks/zuoshu'
    test_output_dir = 'D:/2024/paper2/model/test_results/zuoshu'

    generate_complete_masks(test_input_dir, test_output_dir, generator_path=final_generator_path)
    print("所有测试图像已处理完成。")


Epoch[1/100] Step[0/59], D Loss: 0.7100, G Loss: 99.6853
Epoch[1/100] Step[50/59], D Loss: 0.0727, G Loss: 7.8762
Epoch[2/100] Step[0/59], D Loss: 0.0804, G Loss: 6.4671
Epoch[2/100] Step[50/59], D Loss: 0.7099, G Loss: 2.1241
Epoch[3/100] Step[0/59], D Loss: 0.6860, G Loss: 1.8280
Epoch[3/100] Step[50/59], D Loss: 0.5951, G Loss: 1.4559
Epoch[4/100] Step[0/59], D Loss: 0.5054, G Loss: 2.4168
Epoch[4/100] Step[50/59], D Loss: 0.5587, G Loss: 3.0835
Epoch[5/100] Step[0/59], D Loss: 0.5865, G Loss: 2.0367
Epoch[5/100] Step[50/59], D Loss: 0.4320, G Loss: 2.5984
Epoch[6/100] Step[0/59], D Loss: 0.2163, G Loss: 2.9006
Epoch[6/100] Step[50/59], D Loss: 0.1269, G Loss: 3.4934
Epoch[7/100] Step[0/59], D Loss: 0.0993, G Loss: 3.4684
Epoch[7/100] Step[50/59], D Loss: 0.2650, G Loss: 1.9133
Epoch[8/100] Step[0/59], D Loss: 0.1076, G Loss: 3.3432
Epoch[8/100] Step[50/59], D Loss: 0.1946, G Loss: 3.7056
Epoch[9/100] Step[0/59], D Loss: 0.1477, G Loss: 3.1164
Epoch[9/100] Step[50/59], D Loss: 0.133

C:\Users\admin\AppData\Local\Temp\ipykernel_18884\4229187949.py:249: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  generator.load_state_dict(torch.load(generator_path, map_l

已处理：D:/2024/paper2/model/input_masks/zuoshu\09080080006000_02.png -> D:/2024/paper2/model/test_results/zuoshu\09080080006000_02.png
已处理：D:/2024/paper2/model/input_masks/zuoshu\09080080006000_03.png -> D:/2024/paper2/model/test_results/zuoshu\09080080006000_03.png
已处理：D:/2024/paper2/model/input_masks/zuoshu\09080080006000_05.png -> D:/2024/paper2/model/test_results/zuoshu\09080080006000_05.png
已处理：D:/2024/paper2/model/input_masks/zuoshu\09080080019000_03.png -> D:/2024/paper2/model/test_results/zuoshu\09080080019000_03.png
已处理：D:/2024/paper2/model/input_masks/zuoshu\09080080019000_04.png -> D:/2024/paper2/model/test_results/zuoshu\09080080019000_04.png
已处理：D:/2024/paper2/model/input_masks/zuoshu\09080080019000_05.png -> D:/2024/paper2/model/test_results/zuoshu\09080080019000_05.png
已处理：D:/2024/paper2/model/input_masks/zuoshu\09080080020000_03.png -> D:/2024/paper2/model/test_results/zuoshu\09080080020000_03.png
已处理：D:/2024/paper2/model/input_masks/zuoshu\09080080020000_04.png -> D:/2024

In [1]:
import os
from PIL import Image
import torch

# 确保您已经定义了：
# 1. generator模型结构
# 2. transform（与训练时相同的预处理）
# 3. device
# 4. generator.load_state_dict()和generator.eval()已执行
# 5. save_image函数与之前一致

def save_image(tensor, path):
    # 根据之前定义的save_image函数进行调整
    tensor = tensor.cpu().clone()
    tensor = (tensor * 0.5) + 0.5  # [-1,1] -> [0,1]
    arr = tensor.squeeze(0).numpy()*255
    arr = arr.astype('uint8')
    img = Image.fromarray(arr, mode='L')
    img.save(path)

def generate_complete_masks(input_dir, output_dir, generator, transform, device):
    # 确保输出目录存在
    os.makedirs(output_dir, exist_ok=True)
    generator.eval()

    # 获取目录下所有的图像文件
    image_files = sorted([f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))])

    for filename in image_files:
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)

        # 打开并预处理输入图像
        img = Image.open(input_path).convert('L')
        img_tensor = transform(img).unsqueeze(0).to(device)

        # 生成补全图像
        with torch.no_grad():
            fake_tgt = generator(img_tensor)

        # 保存结果
        save_image(fake_tgt, output_path)
        print(f"已处理：{input_path} -> {output_path}")

if __name__ == "__main__":
    # 假设您已完成训练并有final_generator_path
    final_generator_path = 'D:/2024/paper2/model/checkpoints/generator_final.pth'
    generator.load_state_dict(torch.load(final_generator_path, map_location=device))
    print("已加载最终生成器模型参数。")

    # 定义测试输入目录和输出目录
    test_input_dir = 'D:/2024/paper2/model/test_masks/huashu'
    test_output_dir = 'D:/2024/paper2/model/test_results/huashu'

    # 批量处理所有图像
    generate_complete_masks(test_input_dir, test_output_dir, generator, transform, device)
    print("所有测试图像已完成处理。")


NameError: name 'generator' is not defined